# DistilBERT with SST-2 dataset baseline

In [3]:
# Imports and configure device

import csv
import json
import torch

from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

/home/tc/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [4]:
# Initialize model and validation dataset
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sst2 = load_dataset("stanfordnlp/sst2")
validation = sst2["validation"]

print(f"Validation examples: {len(validation)}")
print(validation[0])

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3057.52it/s]


Validation examples: 872
{'idx': 0, 'sentence': "it 's a charming and often affecting journey . ", 'label': 1}


In [ ]:
# Evaluate pretrained DistilBERT on SST-2 validation data 

model = model.to(device)
model.eval()

batch_size = 32
predictions = []
positive_probs = []

for start in range(0, len(validation), batch_size):
    batch = validation[start:start + batch_size]

    inputs = tokenizer(
        batch["sentence"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)

    predictions.extend(
        probabilities.argmax(dim=-1).cpu().tolist()
    )
    positive_probs.extend(
        probabilities[:, 1].cpu().tolist()
    )

labels = validation["label"]

# validation check
assert len(predictions) == len(validation), (
    "Prediction count does not match validation dataset size"
)

# Accuracy, F1, and Confusion Matrix
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions)
f1_macro = f1_score(labels, predictions, average="macro")
cm = confusion_matrix(labels, predictions, labels=[0, 1])

print(f"Device: {device}")
print(f"Validation examples: {len(labels)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Positive-class F1: {f1:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print("Confusion Matrix:")
print(cm)


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Save results

cwd = Path.cwd()

# Execution from the repository root, the experiments/
# directory, or Colab's default /content directory.
if cwd.name == "experiments":
    results_dir = cwd / "results"
else:
    results_dir = cwd / "experiments" / "results"

results_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving results to: {results_dir}")
results_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "model": model_name,
    "dataset": "stanfordnlp/sst2",
    "split": "validation",
    "num_examples": len(labels),
    "accuracy": accuracy,
    "f1_positive": f1,
    "f1_macro": f1_macro,
    "confusion_matrix": cm.tolist(),
    "batch_size": batch_size,
    "max_length": 128
}

with open(results_dir / "distilbert_sst2_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open(
    results_dir / "distilbert_sst2_predictions.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow([
        "idx", "sentence", "true_label",
        "predicted_label", "positive_probability"
    ])

    for i, example in enumerate(validation):
        writer.writerow([
            example["idx"],
            example["sentence"],
            example["label"],
            predictions[i],
            positive_probs[i]
        ])

print("Evaluation results saved.")